In [1]:
import pandas as pd
import os

In [2]:
images = []

for file in os.listdir("../Images/Root/Original"):
    if file != ".DS_Store":
        images.append(os.path.join("../Images/Root/Original", file))

images

['../Images/Root/Original/tile_37.png',
 '../Images/Root/Original/tile_4.png',
 '../Images/Root/Original/tile_5.png',
 '../Images/Root/Original/tile_26.png',
 '../Images/Root/Original/tile_27.png',
 '../Images/Root/Original/tile_94.png',
 '../Images/Root/Original/tile_95.png',
 '../Images/Root/Original/tile_40.png',
 '../Images/Root/Original/tile_83.png',
 '../Images/Root/Original/tile_82.png',
 '../Images/Root/Original/tile_51.png',
 '../Images/Root/Original/tile_50.png',
 '../Images/Root/Original/tile_61.png',
 '../Images/Root/Original/tile_49.png',
 '../Images/Root/Original/tile_60.png',
 '../Images/Root/Original/tile_71.png',
 '../Images/Root/Original/tile_72.png',
 '../Images/Root/Original/tile_15.png',
 '../Images/Root/Original/tile_28.png',
 '../Images/Root/Original/tile_14.png',
 '../Images/Root/Original/tile_16.png',
 '../Images/Root/Original/tile_17.png',
 '../Images/Root/Original/tile_38.png',
 '../Images/Root/Original/tile_39.png',
 '../Images/Root/Original/tile_106.png']

## Multi-Otsu

In [5]:
from PIL import Image
import skimage as ski
import numpy as np
import pandas as pd
import os

def run_multi_otsu(in_path, k, out_path):
    '''
    Apply multi_otsu threshold clustering to an image

    Args:
        in_path (string): Path to the input image 
        k (integer): Number of clusters
        out_path (string): Path to folder where outputs will be stored
    '''

    # Read the image 
    img = Image.open(in_path)

    # Convert the image to black and white
    img = img.convert("L")

    # Make the image a numpy array
    img_array = np.array(img)

    # Define thresholds     
    thresholds = ski.filters.threshold_multiotsu(img_array, classes = k)

    # Determine regions
    regions = np.digitize(img_array, bins = thresholds)

    # Since python starts counting from 0, add 1 to regions
    regions = regions + 1

    # Write image as pandas data.frame. Reverse image
    end_string = in_path.split("/")[-1].replace(".png", "_multiotsu.txt")
    pd.DataFrame(regions).iloc[::-1].reset_index(drop = True).to_csv(os.path.join(out_path, end_string), sep = "\t")
    return None

In [7]:
for image in images:
    run_multi_otsu(
        in_path = image,
        k = 2,
        out_path = "/Users/degn400/Git_Repos/UnsupervisedSegmentation/Images/Root/Multiotsu_TXT/"
    )

## Binning

In [8]:
from PIL import Image
import numpy as np
import pandas as pd
import os

def run_binning(in_path, k, out_path):
    '''
    Apply bin threshold clustering to an image

    Args:
        in_path (string): Path to the input image 
        k (integer): Number of clusters
        out_path (string): Path to folder where outputs will be stored
    '''

    # Read the image 
    img = Image.open(in_path)

    # Convert the image to black and white
    img = img.convert("L")

    # Make a pandas dataframe of the image
    image_array = np.array(img)

    # Calculate thresholds
    minimum = np.min(image_array)
    maximum = np.max(image_array)
    diff = maximum - minimum
    multiplier = int(diff / k)
    thresholds = [multiplier * (x + 1) + minimum for x in range(k)]

    # Initialize a dataframe to hold clusters
    clusters = np.zeros(image_array.shape) + 1

    # Get indices where condition happens
    for thresh in thresholds[:-1]:
        index1, index2 = np.where(image_array > thresh)
        clusters[index1, index2] += 1

    # Write image as pandas data.frame. Reverse image
    end_string = in_path.split("/")[-1].replace(".png", "_binning.txt")
    pd.DataFrame(clusters.astype(int)).iloc[::-1].reset_index(drop = True).to_csv(os.path.join(out_path, end_string), sep = "\t")
    return None

In [9]:
for image in images:
    run_binning(
        in_path = image,
        k = 2,
        out_path = "/Users/degn400/Git_Repos/UnsupervisedSegmentation/Images/Root/Binning_TXT/"
    )

## pytorch-tip

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
import cv2
import numpy as np

# pytorch-tip script
# code adapted from: https://github.com/kanezaki/pytorch-unsupervised-segmentation/tree/master?tab=readme-ov-file

use_cuda = torch.cuda.is_available()

def pytorchtip_segmentation(image, min_clusters = 3, max_clusters = 16, layers = 2, max_iterations = 50, verbose = False):

    # Define CNN model
    class MyNet(nn.Module):
        def __init__(self,input_dim):
            super(MyNet, self).__init__()
            self.conv1 = nn.Conv2d(input_dim, max_clusters, kernel_size=3, stride=1, padding=1)
            self.bn1 = nn.BatchNorm2d(max_clusters)
            self.conv2 = nn.ModuleList()
            self.bn2 = nn.ModuleList()
            for i in range(layers-1):
                self.conv2.append(nn.Conv2d(max_clusters, max_clusters, kernel_size=3, stride=1, padding=1))
                self.bn2.append(nn.BatchNorm2d(max_clusters))
            self.conv3 = nn.Conv2d(max_clusters, max_clusters, kernel_size=1, stride=1, padding=0)
            self.bn3 = nn.BatchNorm2d(max_clusters)

        def forward(self, x):
            x = self.conv1(x)
            x = F.relu(x)
            x = self.bn1(x)
            for i in range(layers-1):
                x = self.conv2[i](x)
                x = F.relu( x )
                x = self.bn2[i](x)
            x = self.conv3(x)
            x = self.bn3(x)
            return x
    
    # Load image
    im = cv2.imread(image)
    data = torch.from_numpy( np.array([im.transpose((2, 0, 1)).astype('float32')/255.]))
    if use_cuda:
        data = data.cuda()
    data = Variable(data)

    # Train
    model = MyNet(data.size(1))
    if use_cuda:
        model.cuda()
    model.train()

    # Similarity loss definition
    loss_fn = torch.nn.CrossEntropyLoss()

    # Continuity loss definition
    loss_hpy = torch.nn.L1Loss(reduction='mean')
    loss_hpz = torch.nn.L1Loss(reduction='mean')

    HPy_target = torch.zeros(im.shape[0]-1, im.shape[1], max_clusters)
    HPz_target = torch.zeros(im.shape[0], im.shape[1]-1, max_clusters)
    if use_cuda:
        HPy_target = HPy_target.cuda()
        HPz_target = HPz_target.cuda()
    
    optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum=0.9)

    for batch_idx in range(max_iterations):
        
        optimizer.zero_grad()
        output = model(data)[0]
        output = output.permute( 1, 2, 0 ).contiguous().view( -1, max_clusters)

        outputHP = output.reshape( (im.shape[0], im.shape[1], max_clusters))
        HPy = outputHP[1:, :, :] - outputHP[0:-1, :, :]
        HPz = outputHP[:, 1:, :] - outputHP[:, 0:-1, :]
        lhpy = loss_hpy(HPy,HPy_target)
        lhpz = loss_hpz(HPz,HPz_target)

        ignore, target = torch.max(output, 1)
        im_target = target.data.cpu().numpy()
        nLabels = len(np.unique(im_target))

        # loss 
        loss = loss_fn(output, target) + (lhpy + lhpz)
            
        loss.backward()
        optimizer.step()

        if verbose:
            print (batch_idx, '/', max_iterations, '|', ' label num :', nLabels, ' | loss :', loss.item())

        if nLabels <= min_clusters:
            break

    # Pull final model
    final = model(data)[0]
    output = final.permute( 1, 2, 0 ).contiguous().view( -1, max_clusters)
    ignore, target = torch.max(output, 1)
    return target.data.cpu().numpy().reshape((im.shape[0], im.shape[1]))


In [4]:
for image in images:
    seg = pd.DataFrame(pytorchtip_segmentation(image, min_clusters = 2, max_clusters = 2)+1)[::-1]
    outpath = image.replace("Original", "PyTorch_TXT").replace(".png", ".txt")
    seg.to_csv(outpath, sep = "\t", index = False)